In [ ]:
from kalman import wrap_angle
from local_navigation import prox
from motion_control import controller, tracking
from vision import detect_thymio
import numpy as np

v_left = 100
v_right = 100
state = 0       #0=goal tracking, 1=obstacle avoidance
pos_x = 0       #from vision
pos_y = 0       #from vision
angle = 0       #from vision
path = [(0,0), (50,50), (100,100)]  # path to follow  -- maybe pos=path[0]
goal_x = 100    #from global navigation
goal_y = 100    #from global navigation

obstThrL = 10      # low obstacle threshold to switch state 1->0
obstThrH = 20      # high obstacle threshold to switch state 0->1
Gain_angle = 0.5         # gain for angle goal tracking
Gainp = 0.5        # gain for distance to goal
tolerance = 10      # distance tolerance to consider goal reached

@tdmclient.notebook.sync_var
def get_sensor_data():
    global prox_horizontal
    return prox_horizontal.copy()

@tdmclient.notebook.sync_var
def set_speed(left,right):
    global motor_left_target, motor_right_target
    motor_left_target = left
    motor_right_target = right


#goal_x, goal_y = path[-1]  # last waypoint is the goal

while i < len(path):

    heading_x, heading_y = path[i]

    # Get updated position from vision
    pos_x, pos_y, angle = detect_thymio(frame, H)

    # Calculate distance to current waypoint
    angle_diff, drho = tracking(pos_x, pos_y, angle, heading_x, heading_y)

    # Check whether waypoint has been reached
    if drho < tolerance:

        i += 1

        if i >= len(path):
            break

        continue

    # Read proximity sensors
    obst = get_sensor_data()

    if state == 0: 
        # switch from goal tracking to obst avoidance if obstacle detected
        if max(obst) > obstThrH:
            state = 1
    elif state == 1:
        if max(obst) < obstThrL:
            state = 0
            
    if  state == 0 :
        # goal tracking: turn toward the goal
        angle_diff, drho = tracking(pos_x, pos_y, angle, heading_x, heading_y)
        left_cmd, right_cmd = controller(angle_diff, drho, Gain_angle, Gainp, v_left, v_right)
    
    else:
        # obstacle avoidance
        left_cmd, right_cmd = prox(v_left, v_right, obst)

    # Saturate motor commands
    left_cmd = np.clip(left_cmd, -300, 300)
    right_cmd = np.clip(right_cmd, -300, 300)

    # Send commands to motors
    set_speed(left_cmd, right_cmd)

# Stop robot
set_speed(0, 0)
